# CAA2026 Segmentation Model


## Step 1: ENVIRONMENT SETUP


In [ ]:
# ============================================================
# ENVIRONMENT SETUP: Package Installation
# ============================================================
# Install the dependencies required for:
# - Hugging Face SegFormer
# - Zarr / Parquet data access
# - Google Cloud Storage I/O
# - PyTorch training utilities and progress reporting
#
# Run this cell first in a fresh Google Colab environment.
# ============================================================

import sys, subprocess
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'transformers>=4.56,<5', 'zarr', 'gcsfs', 'pyarrow', 'tqdm', 'scikit-learn'
], check=True)
print('✓ Packages installed')

In [ ]:
# ============================================================
# ENVIRONMENT SETUP: Imports, Reproducibility, and Device
# ============================================================
# Import the libraries used throughout the segmentation pipeline,
# set deterministic random seeds where possible, and select CUDA
# automatically when a GPU is available.
# ============================================================

# -----------------------------
# Core Libraries
# -----------------------------
import os, json, math, time, random, warnings, subprocess
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
import zarr
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset
from torch.optim import AdamW
from torch.optim.lr_scheduler import LambdaLR
from sklearn.model_selection import train_test_split
from transformers import SegformerForSemanticSegmentation

# -----------------------------
# Reproducibility and Runtime Device
# -----------------------------
warnings.filterwarnings('ignore')
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = True
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('PyTorch:', torch.__version__)
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available(): print('GPU:', torch.cuda.get_device_name(0))
print('Device:', device)

## Step 2: CONFIGURATION


In [ ]:
# ============================================================================
# CONFIGURATION
# ============================================================================
# Central configuration for the segmentation experiment:
# - Fixed 11-channel input semantics and target mask
# - Train / validation / test fractions
# - SegFormer-B0 and optimization hyperparameters
# - Early stopping and mid-epoch checkpoint behavior
# - Local Colab paths and Google Cloud Storage output paths
#
# IMPORTANT:
# ZARR_CHANNEL_ORDER is part of the model contract. Changing the order
# without changing preprocessing and the first projection layer would
# silently assign the wrong physical variable to each input channel.
# ============================================================================

IMAGE_SIZE = 100
ZARR_CHANNEL_ORDER = ['B2','B3','B4','B8','B11','B12','DEM','Slope','NDVI','NDWI','BSI']
EXPECTED_CHANNEL_ORDER = ['B2','B3','B4','B8','B11','B12','DEM','Slope','NDVI','NDWI','BSI']
assert ZARR_CHANNEL_ORDER == EXPECTED_CHANNEL_ORDER
N_CHANNELS = len(ZARR_CHANNEL_ORDER)
assert N_CHANNELS == 11
LABEL_KEY = 'labels_soft_sigma1'

# -----------------------------
# Dataset Split and Model Hyperparameters
# -----------------------------
TRAIN_FRAC, VAL_FRAC, TEST_FRAC = 0.70, 0.15, 0.15
PRETRAINED_MODEL = 'nvidia/mit-b0'
BATCH_SIZE = 16
NUM_WORKERS = 0
MAX_EPOCHS = 20
ENCODER_LR = 2e-5
NEW_LAYER_LR = 1e-4
WEIGHT_DECAY = 1e-2
WARMUP_EPOCHS = 2
BCE_WEIGHT, DICE_WEIGHT = 0.30, 0.70
MAX_SHIFT = 32
USE_AMP = True
GRAD_CLIP_NORM = 1.0

# Early stop only resets after a meaningful val-loss improvement.
EARLY_STOP_MIN_DELTA = 0.001
EARLY_STOP_PATIENCE = 3

# Mid-epoch recovery granularity.
CHECKPOINT_EVERY_N_BATCHES = 100
RESUME_IF_AVAILABLE = True

# -----------------------------
# Reproducible Visualization
# -----------------------------
FIXED_VIS_SEED = 123
N_VIS_SAMPLES = 6

# -----------------------------
# Input Data Locations
# -----------------------------
GCS_DATA_ROOT = 'gs://caa-2026/dataset'
GCS_ZARR_POS = f'{GCS_DATA_ROOT}/positives.zarr'
GCS_ZARR_META_POS = f'{GCS_DATA_ROOT}/metadata/positives.parquet'

LOCAL_STAGE2_ROOT = '/content/stage2_segmentation'
LOCAL_ZARR_DIR = f'{LOCAL_STAGE2_ROOT}/zarr_data'
LOCAL_META_DIR = f'{LOCAL_STAGE2_ROOT}/metadata'
LOCAL_POS_ZARR = f'{LOCAL_ZARR_DIR}/positives.zarr'
LOCAL_POS_META = f'{LOCAL_META_DIR}/positives.parquet'

# -----------------------------
# Experiment Output Locations
# -----------------------------
RUN_NAME = 'segformer_b0_11ch_sigma1' # change the version name here
LOCAL_RUN_ROOT = f'/content/{RUN_NAME}'
LOCAL_CKPT_DIR = f'{LOCAL_RUN_ROOT}/checkpoints'
LOCAL_REPORT_DIR = f'{LOCAL_RUN_ROOT}/reports'
LOCAL_FIG_DIR = f'{LOCAL_RUN_ROOT}/figures'
LOCAL_HF_DIR = f'{LOCAL_RUN_ROOT}/hf_model'
for p in [LOCAL_STAGE2_ROOT, LOCAL_ZARR_DIR, LOCAL_META_DIR, LOCAL_RUN_ROOT, LOCAL_CKPT_DIR, LOCAL_REPORT_DIR, LOCAL_FIG_DIR, LOCAL_HF_DIR]:
    os.makedirs(p, exist_ok=True)

GCS_RUN_ROOT = f'gs://caa-2026/models/{RUN_NAME}'
GCS_CKPT_DIR = f'{GCS_RUN_ROOT}/checkpoints'
GCS_REPORT_DIR = f'{GCS_RUN_ROOT}/reports'
GCS_FIG_DIR = f'{GCS_RUN_ROOT}/figures'
GCS_HF_DIR = f'{GCS_RUN_ROOT}/hf_model'

# -----------------------------
# Checkpoint Paths
# -----------------------------
RESUME_CKPT_LOCAL = f'{LOCAL_CKPT_DIR}/resume_checkpoint.pt'
BEST_CKPT_LOCAL = f'{LOCAL_CKPT_DIR}/best_checkpoint.pt'
RESUME_CKPT_GCS = f'{GCS_CKPT_DIR}/resume_checkpoint.pt'
BEST_CKPT_GCS = f'{GCS_CKPT_DIR}/best_checkpoint.pt'

print('✓ Config loaded')
print('Input: 11 × 100 × 100')
print('Channels:', ZARR_CHANNEL_ORDER)
print('Target:', LABEL_KEY)
print('Max epochs:', MAX_EPOCHS)
print(f'Early stop: min_delta={EARLY_STOP_MIN_DELTA}, patience={EARLY_STOP_PATIENCE}')
print('Mid-epoch checkpoint every', CHECKPOINT_EVERY_N_BATCHES, 'batches')
print('GCS output:', GCS_RUN_ROOT)

## Step 3: DATA IMPORT / DOWNLOAD / LOCAL PREPARATION


In [ ]:
# ============================================================
# DATA IMPORT / DOWNLOAD / LOCAL PREPARATION
# ============================================================
# This section:
# 1. Authenticates Google Colab with Google Cloud
# 2. Defines reusable GCS helper functions
# 3. Downloads the positive Zarr store and Parquet metadata if needed
# 4. Opens the existing arrays and validates all required mask keys
#
# Existing local copies are reused to avoid unnecessary downloads.
# ============================================================

from google.colab import auth
auth.authenticate_user()
print('✓ Google authentication complete')

# -----------------------------
# Google Cloud Storage Helpers
# -----------------------------
def run_cmd(cmd, quiet=False, check=True):
    r = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    if not quiet and r.stdout.strip(): print(r.stdout.strip())
    if r.returncode != 0:
        if not quiet and r.stderr.strip(): print(r.stderr.strip())
        if check: raise RuntimeError(f'Command failed: {cmd}')
    return r

def gcs_exists(gcs_path):
    return run_cmd(f'gsutil -q stat "{gcs_path}"', quiet=True, check=False).returncode == 0

def gcs_upload(local_path, gcs_path, quiet=True):
    return run_cmd(f'gsutil cp "{local_path}" "{gcs_path}"', quiet=quiet)

def gcs_download_file(gcs_path, local_path):
    os.makedirs(os.path.dirname(local_path), exist_ok=True)
    return run_cmd(f'gsutil cp "{gcs_path}" "{local_path}"', quiet=True)

def gcs_rsync(local_dir, gcs_dir):
    return run_cmd(f'gsutil -m rsync -r "{local_dir}" "{gcs_dir}"', quiet=False)

# -----------------------------
# Download Dataset Assets
# -----------------------------
if not os.path.exists(LOCAL_POS_ZARR):
    print('Downloading positives.zarr ...')
    t0=time.time(); run_cmd(f'gsutil -m cp -r "{GCS_ZARR_POS}" "{LOCAL_ZARR_DIR}"')
    print(f'✓ positives.zarr downloaded in {time.time()-t0:.1f}s')
else:
    print('✓ positives.zarr already local')

if not os.path.exists(LOCAL_POS_META):
    print('Downloading positives.parquet ...')
    run_cmd(f'gsutil cp "{GCS_ZARR_META_POS}" "{LOCAL_META_DIR}"')
else:
    print('✓ positives.parquet already local')

# -----------------------------
# Open and Validate Zarr Data
# -----------------------------
pos_root = zarr.open(LOCAL_POS_ZARR, mode='r')
pos_meta = pd.read_parquet(LOCAL_POS_META)
for key in ['grids','labels_hard','labels_soft_sigma1','labels_soft_sigma3','labels_soft_sigma8']:
    assert key in pos_root, f'Missing {key}'
pos_grids = pos_root['grids']
pos_labels_hard = pos_root['labels_hard']
pos_labels_sigma1 = pos_root['labels_soft_sigma1']
print('✓ Zarr opened')
print('Rows:', len(pos_meta))
print('grids:', pos_grids.shape, pos_grids.dtype)
print('sigma1:', pos_labels_sigma1.shape)
print('hard:', pos_labels_hard.shape)
print('Resume checkpoint on GCS:', gcs_exists(RESUME_CKPT_GCS))

In [ ]:
# ============================================================
# ZARR ARRAY READERS & CHANNEL-ORDER VALIDATION
# ============================================================
# These helper functions read one image or mask by zarr_index.
#
# The image reader accepts either CHW or HWC storage layouts, but only
# moves the channel axis; it NEVER changes the semantic channel order.
# This preserves the fixed input contract:
# [B2, B3, B4, B8, B11, B12, DEM, Slope, NDVI, NDWI, BSI].
# ============================================================

def read_pos_grid_by_index(zarr_index: int):
    x = np.asarray(pos_grids[int(zarr_index)]).astype(np.float32)
    if x.ndim != 3: raise ValueError(f'Expected 3D grid, got {x.shape}')
    if x.shape[0] == N_CHANNELS:
        out = x
    elif x.shape[-1] == N_CHANNELS:
        out = np.transpose(x, (2,0,1))  # axis move only; semantic channel order unchanged
    else:
        raise ValueError(f'Cannot identify channel axis: {x.shape}')
    assert out.shape == (N_CHANNELS, IMAGE_SIZE, IMAGE_SIZE)
    return out

def read_soft_label_by_index(zarr_index: int):
    y = np.asarray(pos_labels_sigma1[int(zarr_index)]).astype(np.float32)
    assert y.shape == (IMAGE_SIZE, IMAGE_SIZE)
    return y

def read_hard_label_by_index(zarr_index: int):
    y = np.asarray(pos_labels_hard[int(zarr_index)]).astype(np.float32)
    assert y.shape == (IMAGE_SIZE, IMAGE_SIZE)
    return y

_ = read_pos_grid_by_index(int(pos_meta.iloc[0]['zarr_index']))
print('✓ Fixed channel semantics preserved:')
for i,ch in enumerate(ZARR_CHANNEL_ORDER): print(f'  {i:2d}: {ch}')

## Step 4: LEAKAGE-SAFE GROUPED SPLIT


In [ ]:
# ============================================================
# LEAKAGE-SAFE GROUPED SPLIT
# ============================================================
# Create a reproducible train / validation / test split at the GROUP level.
#
# Grouping priority:
# - Use site_id when available
# - Otherwise fall back to grid_id
#
# CRITICAL:
# Samples from the same archaeological site are kept in exactly one split.
# This prevents spatially related samples from leaking across train,
# validation, and test sets and inflating evaluation performance.
#
# If a saved split already exists on GCS, it is reused for resume consistency.
# ============================================================

def infer_group(row):
    site_id = str(row.get('site_id','')).strip()
    if site_id and site_id.lower() != 'nan': return f'site::{site_id}'
    return f"pos_grid::{str(row.get('grid_id','')).strip()}"

GCS_SPLIT_CSV=f'{GCS_REPORT_DIR}/split.csv'
LOCAL_SPLIT_CSV=f'{LOCAL_REPORT_DIR}/split.csv'

# -----------------------------
# Reuse Existing Split or Create a New One
# -----------------------------
if RESUME_IF_AVAILABLE and gcs_exists(GCS_SPLIT_CSV):
    gcs_download_file(GCS_SPLIT_CSV, LOCAL_SPLIT_CSV)
    stage2_df = pd.read_csv(LOCAL_SPLIT_CSV)
    print('✓ Loaded existing split from GCS')
else:
    stage2_df = pos_meta.copy()
    stage2_df['sample_id'] = stage2_df['grid_id'].astype(str)
    stage2_df['group_id'] = stage2_df.apply(infer_group, axis=1)
    stage2_df['zarr_index'] = pd.to_numeric(stage2_df['zarr_index'], errors='coerce').astype(int)

    groups = np.array(sorted(stage2_df['group_id'].astype(str).unique()))
    train_groups, temp_groups = train_test_split(groups, test_size=1-TRAIN_FRAC, random_state=SEED, shuffle=True)
    val_ratio = VAL_FRAC/(VAL_FRAC+TEST_FRAC)
    val_groups, test_groups = train_test_split(temp_groups, test_size=1-val_ratio, random_state=SEED, shuffle=True)
    train_groups, val_groups, test_groups = set(train_groups), set(val_groups), set(test_groups)

    def assign_split(g):
        g=str(g)
        if g in train_groups: return 'train'
        if g in val_groups: return 'val'
        if g in test_groups: return 'test'
        return 'unknown'

    stage2_df['split'] = stage2_df['group_id'].apply(assign_split)
    assert (stage2_df['split']=='unknown').sum()==0
    stage2_df.to_csv(LOCAL_SPLIT_CSV,index=False)
    gcs_upload(LOCAL_SPLIT_CSV, GCS_SPLIT_CSV)
    print('✓ Created split and saved to GCS')

# -----------------------------
# Materialize Splits and Verify No Group Overlap
# -----------------------------
stage2_train_df = stage2_df[stage2_df['split']=='train'].reset_index(drop=True)
stage2_val_df = stage2_df[stage2_df['split']=='val'].reset_index(drop=True)
stage2_test_df = stage2_df[stage2_df['split']=='test'].reset_index(drop=True)
assert set(stage2_train_df.group_id).isdisjoint(set(stage2_val_df.group_id))
assert set(stage2_train_df.group_id).isdisjoint(set(stage2_test_df.group_id))
assert set(stage2_val_df.group_id).isdisjoint(set(stage2_test_df.group_id))
print('✓ Split ready; no group leakage')
print('Train / Val / Test:', len(stage2_train_df), len(stage2_val_df), len(stage2_test_df))

## Step 5: CHANNEL STATISTICS & TRAIN-ONLY NORMALIZATION


In [ ]:
# ============================================================
# CHANNEL STATISTICS & TRAIN-ONLY NORMALIZATION
# ============================================================
# This section:
# 1. Converts raw optical channels to reflectance-scale values
# 2. Computes channel-wise mean/std using TRAINING samples only
# 3. Saves normalization parameters for reproducible resume/inference
# 4. Applies z-score normalization to all 11 input channels
# ============================================================

OPTICAL_IDX=[0,1,2,3,4,5]; DEM_IDX=[6]; SLOPE_IDX=[7]; INDEX_IDX=[8,9,10]
CHANNEL_TO_INDEX={ch:i for i,ch in enumerate(ZARR_CHANNEL_ORDER)}

# -----------------------------
# Physical Unit Conversion
# -----------------------------
def raw_to_physical(x_raw):
    x=x_raw.astype(np.float32); out=np.zeros_like(x,dtype=np.float32)
    out[OPTICAL_IDX]=x[OPTICAL_IDX]/10000.0
    out[DEM_IDX]=x[DEM_IDX]; out[SLOPE_IDX]=x[SLOPE_IDX]; out[INDEX_IDX]=x[INDEX_IDX]
    return out

GCS_NORM_JSON=f'{GCS_REPORT_DIR}/normalization_params.json'
LOCAL_NORM_JSON=f'{LOCAL_REPORT_DIR}/normalization_params.json'

# -----------------------------
# Load Existing Stats or Compute from Training Data
# -----------------------------
if RESUME_IF_AVAILABLE and gcs_exists(GCS_NORM_JSON):
    gcs_download_file(GCS_NORM_JSON, LOCAL_NORM_JSON)
    with open(LOCAL_NORM_JSON) as f: stage2_norm_params=json.load(f)
    print('✓ Loaded normalization parameters from GCS')
else:
    MAX_STATS_SAMPLES=3000; PIXELS_PER_IMAGE_FOR_STATS=2048
    stats_df = stage2_train_df.sample(n=min(MAX_STATS_SAMPLES,len(stage2_train_df)), random_state=SEED).reset_index(drop=True)
    rng_stats=np.random.default_rng(SEED)
    sum_c=np.zeros(N_CHANNELS,np.float64); sumsq_c=np.zeros(N_CHANNELS,np.float64); count_c=np.zeros(N_CHANNELS,np.int64)
    print(f'Computing train-only normalization from {len(stats_df):,} samples ...')
    for _,row in tqdm(stats_df.iterrows(), total=len(stats_df), desc='Normalization stats'):
        x=raw_to_physical(read_pos_grid_by_index(int(row.zarr_index)))
        for ci in range(N_CHANNELS):
            flat=x[ci].reshape(-1)
            if flat.size>PIXELS_PER_IMAGE_FOR_STATS:
                flat=flat[rng_stats.choice(flat.size, PIXELS_PER_IMAGE_FOR_STATS, replace=False)]
            vals=flat[np.isfinite(flat)].astype(np.float64)
            if vals.size:
                sum_c[ci]+=vals.sum(); sumsq_c[ci]+=np.square(vals).sum(); count_c[ci]+=vals.size
    means=sum_c/np.maximum(count_c,1)
    vars_=sumsq_c/np.maximum(count_c,1)-means**2
    stds=np.sqrt(np.maximum(vars_,1e-12))
    stage2_norm_params={ch:{'mean':float(means[i]),'std':float(stds[i] if stds[i]>1e-8 else 1.0)} for i,ch in enumerate(ZARR_CHANNEL_ORDER)}
    with open(LOCAL_NORM_JSON,'w') as f: json.dump(stage2_norm_params,f,indent=2)
    gcs_upload(LOCAL_NORM_JSON, GCS_NORM_JSON)
    print('✓ Computed normalization and saved to GCS')

# -----------------------------
# Apply Saved Channel-wise Z-score Normalization
# -----------------------------
def normalize_stage2(x_raw):
    x=raw_to_physical(x_raw); out=np.zeros_like(x,dtype=np.float32)
    for ch,ci in CHANNEL_TO_INDEX.items():
        p=stage2_norm_params[ch]
        out[ci]=np.nan_to_num((x[ci]-p['mean'])/p['std'], nan=0.0, posinf=0.0, neginf=0.0)
    return out

display(pd.DataFrame([{'channel':ch,**stage2_norm_params[ch]} for ch in ZARR_CHANNEL_ORDER]))

## Step 6: DATASET & DATALOADER IMPLEMENTATION


In [ ]:
# ============================================================
# DATASET & DATALOADER IMPLEMENTATION
# ============================================================
# This section:
# 1. Defines spatial augmentations shared by images and masks
# 2. Implements a deterministic PyTorch Dataset
# 3. Uses soft masks for optimization and hard masks for evaluation
# 4. Builds validation/test loaders
# 5. Creates a reproducible per-epoch training order that supports
#    resuming from the middle of an epoch
#
# Deterministic augmentation is important here because a resumed run should
# see the same transformed sample for a given (epoch, sample index).
# ============================================================

def shift_chw_reflect(x,dy,dx,max_shift):
    c,h,w=x.shape; p=int(max_shift)
    xp=np.pad(x,((0,0),(p,p),(p,p)),mode='reflect')
    y0=p-int(dy); x0=p-int(dx)
    out=xp[:,y0:y0+h,x0:x0+w].copy(); assert out.shape==x.shape
    return out

def shift_hw_zero(y,dy,dx):
    h,w=y.shape; out=np.zeros_like(y)
    sy0=max(0,-dy); sy1=min(h,h-dy) if dy>=0 else h; dy0=max(0,dy); dy1=dy0+(sy1-sy0)
    sx0=max(0,-dx); sx1=min(w,w-dx) if dx>=0 else w; dx0=max(0,dx); dx1=dx0+(sx1-sx0)
    if sy1>sy0 and sx1>sx0: out[dy0:dy1,dx0:dx1]=y[sy0:sy1,sx0:sx1]
    return out

# -----------------------------
# Deterministic Spatial Augmentation
# -----------------------------
def deterministic_augment(x,soft,hard,seed):
    rng=np.random.default_rng(seed)
    if rng.random()<0.5: x=np.flip(x,2).copy(); soft=np.flip(soft,1).copy(); hard=np.flip(hard,1).copy()
    if rng.random()<0.5: x=np.flip(x,1).copy(); soft=np.flip(soft,0).copy(); hard=np.flip(hard,0).copy()
    k=int(rng.integers(0,4))
    if k: x=np.rot90(x,k,axes=(1,2)).copy(); soft=np.rot90(soft,k).copy(); hard=np.rot90(hard,k).copy()
    dy=int(rng.integers(-MAX_SHIFT,MAX_SHIFT+1)); dx=int(rng.integers(-MAX_SHIFT,MAX_SHIFT+1))
    return shift_chw_reflect(x,dy,dx,MAX_SHIFT), shift_hw_zero(soft,dy,dx), shift_hw_zero(hard,dy,dx)

# -----------------------------
# PyTorch Dataset
# -----------------------------
class GeoglyphDataset(Dataset):
    def __init__(self,df,augment=False,seed=42):
        self.df=df.reset_index(drop=True).copy(); self.augment=augment; self.base_seed=int(seed); self.epoch=0
    def set_epoch(self,epoch): self.epoch=int(epoch)
    def __len__(self): return len(self.df)
    def __getitem__(self,idx):
        idx=int(idx); row=self.df.iloc[idx]; zidx=int(row.zarr_index)
        x=normalize_stage2(read_pos_grid_by_index(zidx))
        soft=read_soft_label_by_index(zidx); hard=read_hard_label_by_index(zidx)
        if self.augment:
            aug_seed=self.base_seed+self.epoch*1000003+idx*9973
            x,soft,hard=deterministic_augment(x,soft,hard,aug_seed)
        return {'image':torch.from_numpy(x).float(),'mask_soft':torch.from_numpy(soft[None]).float(),'mask_hard':torch.from_numpy(hard[None]).float(),'meta':{'sample_id':str(row.sample_id),'zarr_index':zidx,'split':str(row.split)}}

# -----------------------------
# Validation and Test DataLoaders
# -----------------------------
train_ds=GeoglyphDataset(stage2_train_df,True,SEED); val_ds=GeoglyphDataset(stage2_val_df,False,SEED); test_ds=GeoglyphDataset(stage2_test_df,False,SEED)
val_loader=DataLoader(val_ds,batch_size=BATCH_SIZE,shuffle=False,num_workers=NUM_WORKERS,pin_memory=torch.cuda.is_available())
test_loader=DataLoader(test_ds,batch_size=BATCH_SIZE,shuffle=False,num_workers=NUM_WORKERS,pin_memory=torch.cuda.is_available())

# -----------------------------
# Resume-Safe Training Sample Order
# -----------------------------
def epoch_sample_order(epoch):
    rng=np.random.default_rng(SEED+int(epoch)*104729)
    return rng.permutation(len(train_ds)).tolist()

def make_train_loader_for_epoch(epoch,start_batch=0):
    train_ds.set_epoch(epoch)
    order=epoch_sample_order(epoch)
    start_sample=int(start_batch)*BATCH_SIZE
    remaining=order[start_sample:]
    loader_gen=torch.Generator(); loader_gen.manual_seed(SEED+int(epoch)*1009)
    loader=DataLoader(Subset(train_ds,remaining),batch_size=BATCH_SIZE,shuffle=False,num_workers=NUM_WORKERS,pin_memory=torch.cuda.is_available(),generator=loader_gen)
    return loader, math.ceil(len(train_ds)/BATCH_SIZE)

sample=train_ds[0]
print('✓ Deterministic dataset ready')
print('Train / Val / Test:',len(train_ds),len(val_ds),len(test_ds))
print('Image:',tuple(sample['image'].shape),'Soft:',tuple(sample['mask_soft'].shape),'Hard:',tuple(sample['mask_hard'].shape))

## Step 7: MODEL ARCHITECTURE


In [ ]:
# ============================================================
# MODEL ARCHITECTURE: 11-CHANNEL SEGFORMER-B0
# ============================================================
# Start from the ImageNet-pretrained SegFormer-B0 backbone and adapt it
# from the standard 3-channel RGB input to the 11-channel geospatial input.
#
# Initialization strategy for the first projection:
# - B2 (Blue)  <- pretrained RGB Blue weights
# - B3 (Green) <- pretrained RGB Green weights
# - B4 (Red)   <- pretrained RGB Red weights
# - Remaining 8 channels start at zero in the first projection
#
# The segmentation head predicts one logit map for binary geoglyph masks.
# ============================================================

model=SegformerForSemanticSegmentation.from_pretrained(PRETRAINED_MODEL,num_labels=1,id2label={0:'geoglyph'},label2id={'geoglyph':0},ignore_mismatched_sizes=True)

# -----------------------------
# Expand First Patch Projection: 3 Channels -> 11 Channels
# -----------------------------
def expand_first_projection(model):
    old=model.segformer.encoder.patch_embeddings[0].proj
    assert old.in_channels==3
    new=nn.Conv2d(N_CHANNELS,old.out_channels,kernel_size=old.kernel_size,stride=old.stride,padding=old.padding,dilation=old.dilation,groups=old.groups,bias=old.bias is not None,padding_mode=old.padding_mode).to(old.weight.device,dtype=old.weight.dtype)
    with torch.no_grad():
        new.weight.zero_()
        new.weight[:,0]=old.weight[:,2]  # fixed B2 gets pretrained Blue
        new.weight[:,1]=old.weight[:,1]  # fixed B3 gets pretrained Green
        new.weight[:,2]=old.weight[:,0]  # fixed B4 gets pretrained Red
        if old.bias is not None: new.bias.copy_(old.bias)
    model.segformer.encoder.patch_embeddings[0].proj=new
    model.config.num_channels=N_CHANNELS
    return model

# -----------------------------
# Build Model and Run a Forward-Pass Sanity Check
# -----------------------------
model=expand_first_projection(model).to(device)
assert model.segformer.encoder.patch_embeddings[0].proj.in_channels==11
print('✓ SegFormer-B0 ready')
print('Trainable params:',f"{sum(p.numel() for p in model.parameters() if p.requires_grad):,}")
with torch.no_grad():
    x0=sample['image'].unsqueeze(0).to(device); y0=model(pixel_values=x0).logits; y0=F.interpolate(y0,size=(100,100),mode='bilinear',align_corners=False)
print('Forward sanity:',tuple(x0.shape),'->',tuple(y0.shape))

## Step 8: LOSS, METRICS & OPTIMIZER


In [ ]:
# ============================================================
# LOSS, METRICS, OPTIMIZER & LR SCHEDULER
# ============================================================
# Training objective:
# - Binary cross-entropy with logits for pixel-wise supervision
# - Soft Dice loss for region overlap
# - Weighted combination controlled by BCE_WEIGHT / DICE_WEIGHT
#
# Evaluation:
# - Dice and IoU are computed against HARD masks after thresholding
# - Optional centroid error measures localization error in pixels
#
# Optimization:
# - Lower learning rate for the pretrained encoder
# - Higher learning rate for the new 11-channel projection and decode head
# - Linear warmup followed by cosine decay
# ============================================================

def forward_logits(images):
    logits=model(pixel_values=images).logits
    return F.interpolate(logits,size=(IMAGE_SIZE,IMAGE_SIZE),mode='bilinear',align_corners=False)

# -----------------------------
# Segmentation Loss
# -----------------------------
def soft_dice_loss(logits,target,eps=1e-6):
    p=torch.sigmoid(logits); dims=(1,2,3); inter=(p*target).sum(dims); den=p.sum(dims)+target.sum(dims)
    return 1-((2*inter+eps)/(den+eps)).mean()

def seg_loss(logits,target):
    bce=F.binary_cross_entropy_with_logits(logits,target); dl=soft_dice_loss(logits,target)
    return BCE_WEIGHT*bce+DICE_WEIGHT*dl

# -----------------------------
# Segmentation and Localization Metrics
# -----------------------------
def dice_iou(probs,hard,threshold=0.5,eps=1e-6):
    pred=(probs>=threshold).float(); target=(hard>=0.5).float(); dims=(1,2,3)
    inter=(pred*target).sum(dims); ps=pred.sum(dims); ts=target.sum(dims)
    dice=(2*inter+eps)/(ps+ts+eps); iou=(inter+eps)/(ps+ts-inter+eps)
    return dice,iou

def centroid_errors(pred,target):
    p=pred.detach().cpu().numpy(); t=target.detach().cpu().numpy(); out=[]
    for i in range(len(p)):
        py,px=np.where(p[i,0]>0.5); ty,tx=np.where(t[i,0]>0.5)
        if len(px)==0 or len(tx)==0: out.append(np.nan); continue
        out.append(float(math.hypot(px.mean()-tx.mean(),py.mean()-ty.mean())))
    return out

@torch.no_grad()
def evaluate(loader,threshold=0.5,with_centroid=False):
    model.eval(); loss_sum=dice_sum=iou_sum=0.; n=0; cents=[]; nonempty=0
    for batch in loader:
        x=batch['image'].to(device); soft=batch['mask_soft'].to(device); hard=batch['mask_hard'].to(device)
        logits=forward_logits(x); loss=seg_loss(logits,soft); probs=torch.sigmoid(logits); d,i=dice_iou(probs,hard,threshold)
        bs=len(x); n+=bs; loss_sum+=loss.item()*bs; dice_sum+=d.sum().item(); iou_sum+=i.sum().item()
        if with_centroid:
            pb=(probs>=threshold).float(); hb=(hard>=0.5).float(); nonempty+=(pb.sum((1,2,3))>0).sum().item(); cents+=centroid_errors(pb,hb)
    r={'loss':loss_sum/max(n,1),'dice':dice_sum/max(n,1),'iou':iou_sum/max(n,1)}
    if with_centroid:
        finite=[v for v in cents if np.isfinite(v)]; r['centroid_error_px_mean']=float(np.mean(finite)) if finite else np.nan; r['centroid_error_px_median']=float(np.median(finite)) if finite else np.nan; r['pred_nonempty_fraction']=float(nonempty/max(n,1))
    return r

# -----------------------------
# Differential Learning Rates and Scheduler
# -----------------------------
input_params=list(model.segformer.encoder.patch_embeddings[0].proj.parameters()); head_params=list(model.decode_head.parameters()); new_ids={id(p) for p in input_params+head_params}; enc_params=[p for p in model.parameters() if id(p) not in new_ids]
optimizer=AdamW([{'params':enc_params,'lr':ENCODER_LR},{'params':input_params,'lr':NEW_LAYER_LR},{'params':head_params,'lr':NEW_LAYER_LR}],weight_decay=WEIGHT_DECAY)
FULL_BATCHES_PER_EPOCH=math.ceil(len(train_ds)/BATCH_SIZE)
total_steps=max(1,MAX_EPOCHS*FULL_BATCHES_PER_EPOCH); warmup_steps=min(total_steps-1,WARMUP_EPOCHS*FULL_BATCHES_PER_EPOCH)
def lr_lambda(step):
    if warmup_steps and step<warmup_steps: return (step+1)/warmup_steps
    p=min(max((step-warmup_steps)/max(1,total_steps-warmup_steps),0),1); return 0.5*(1+math.cos(math.pi*p))
scheduler=LambdaLR(optimizer,lr_lambda)
amp_enabled=USE_AMP and device.type=='cuda'; scaler=torch.cuda.amp.GradScaler(enabled=amp_enabled)
print('✓ Training components ready | AMP:',amp_enabled)
print('Full batches per epoch:',FULL_BATCHES_PER_EPOCH)

## Step 9: RESUME-SAFE TRAINING LOOP


In [ ]:
# ============================================================
# RESUME-SAFE TRAINING LOOP & GCS CHECKPOINTING
# ============================================================
# This section implements fault-tolerant training designed for Colab/GPU
# interruptions while preserving reproducibility.
#
# Checkpoint strategy:
# - BEST checkpoint: model state from the lowest validation loss
# - RESUME checkpoint: complete training state, including optimizer,
#   scheduler, AMP scaler, RNG states, epoch, and next batch position
# - Mid-epoch resume checkpoints are uploaded every N batches
#
# Early stopping:
# Patience resets only when validation loss improves by at least
# EARLY_STOP_MIN_DELTA, avoiding resets from negligible fluctuations.
# ============================================================

run_config={'run_name':RUN_NAME,'model':PRETRAINED_MODEL,'image_size':IMAGE_SIZE,'channel_order':ZARR_CHANNEL_ORDER,'label_key':LABEL_KEY,'batch_size':BATCH_SIZE,'max_epochs':MAX_EPOCHS,'encoder_lr':ENCODER_LR,'new_layer_lr':NEW_LAYER_LR,'loss':{'bce':BCE_WEIGHT,'dice':DICE_WEIGHT},'max_shift':MAX_SHIFT,'early_stop_min_delta':EARLY_STOP_MIN_DELTA,'early_stop_patience':EARLY_STOP_PATIENCE,'checkpoint_every_n_batches':CHECKPOINT_EVERY_N_BATCHES}
config_path=f'{LOCAL_REPORT_DIR}/run_config.json'
with open(config_path,'w') as f: json.dump(run_config,f,indent=2)
gcs_upload(config_path,f'{GCS_REPORT_DIR}/run_config.json')

# -----------------------------
# Robust Checkpoint Loading and RNG State Management
# -----------------------------
def safe_load(path):
    try:
        return torch.load(
            path,
            map_location="cpu",
            weights_only=False,
        )
    except TypeError:
        return torch.load(
            path,
            map_location="cpu",
        )

def capture_rng_state():
    s={'python':random.getstate(),'numpy':np.random.get_state(),'torch':torch.get_rng_state()}
    if torch.cuda.is_available(): s['cuda']=torch.cuda.get_rng_state_all()
    return s

def restore_rng_state(s):
    if not s:
        return

    if "python" in s:
        random.setstate(s["python"])

    if "numpy" in s:
        np.random.set_state(s["numpy"])

    if "torch" in s:
        torch_state = s["torch"].detach().cpu().to(torch.uint8)
        torch.set_rng_state(torch_state)

    if torch.cuda.is_available() and "cuda" in s:
        cuda_states = [
            state.detach().cpu().to(torch.uint8)
            for state in s["cuda"]
        ]
        torch.cuda.set_rng_state_all(cuda_states)

# -----------------------------
# Save Resume and Best Checkpoints
# -----------------------------
def save_resume(epoch_index,next_batch,global_step,epoch_loss_sum,epoch_sample_count,history,best_val,best_epoch,early_ref,stale):
    payload={'format_version':2,'epoch_index':int(epoch_index),'next_batch':int(next_batch),'global_step':int(global_step),'epoch_loss_sum':float(epoch_loss_sum),'epoch_sample_count':int(epoch_sample_count),'history':history,'best_val_loss':float(best_val),'best_epoch':int(best_epoch),'early_stop_reference':float(early_ref),'stale_epochs':int(stale),'model_state_dict':model.state_dict(),'optimizer_state_dict':optimizer.state_dict(),'scheduler_state_dict':scheduler.state_dict(),'scaler_state_dict':scaler.state_dict(),'rng_state':capture_rng_state(),'channel_order':ZARR_CHANNEL_ORDER,'normalization_params':stage2_norm_params,'run_config':run_config}
    torch.save(payload,RESUME_CKPT_LOCAL); gcs_upload(RESUME_CKPT_LOCAL,RESUME_CKPT_GCS,quiet=True)

def save_best(epoch,val):
    torch.save({'epoch':int(epoch),'val_metrics':val,'model_state_dict':model.state_dict(),'channel_order':ZARR_CHANNEL_ORDER,'normalization_params':stage2_norm_params,'run_config':run_config},BEST_CKPT_LOCAL)
    gcs_upload(BEST_CKPT_LOCAL,BEST_CKPT_GCS,quiet=True)

# -----------------------------
# Restore Existing Run State When Available
# -----------------------------
start_epoch=0; start_batch=0; global_step=0; epoch_loss_sum_resume=0.; epoch_sample_count_resume=0
history=[]; best_val=float('inf'); best_epoch=-1; early_ref=float('inf'); stale=0

if RESUME_IF_AVAILABLE and gcs_exists(RESUME_CKPT_GCS):
    print('Resume checkpoint found on GCS — restoring ...')
    gcs_download_file(RESUME_CKPT_GCS,RESUME_CKPT_LOCAL); r=safe_load(RESUME_CKPT_LOCAL)
    assert r['run_config']['channel_order']==ZARR_CHANNEL_ORDER and r['run_config']['image_size']==IMAGE_SIZE and r['run_config']['label_key']==LABEL_KEY
    model.load_state_dict(r['model_state_dict']); optimizer.load_state_dict(r['optimizer_state_dict']); scheduler.load_state_dict(r['scheduler_state_dict']); scaler.load_state_dict(r['scaler_state_dict']); restore_rng_state(r.get('rng_state'))
    start_epoch=int(r['epoch_index']); start_batch=int(r['next_batch']); global_step=int(r['global_step']); epoch_loss_sum_resume=float(r['epoch_loss_sum']); epoch_sample_count_resume=int(r['epoch_sample_count']); history=list(r.get('history',[])); best_val=float(r['best_val_loss']); best_epoch=int(r['best_epoch']); early_ref=float(r['early_stop_reference']); stale=int(r['stale_epochs'])
    print(f'✓ Resume position: epoch {start_epoch+1}, batch {start_batch}/{FULL_BATCHES_PER_EPOCH}')
    print('Best epoch so far:',best_epoch,'| best val loss:',best_val,'| stale:',stale)
else:
    print('✓ No resume checkpoint found — starting fresh')

# -----------------------------
# Main Training / Validation Loop
# -----------------------------
print('============================================================')
print('Starting SegFormer-B0 training')
print('GCS:',GCS_RUN_ROOT)
print('============================================================')

loop_start_epoch=start_epoch
if stale>=EARLY_STOP_PATIENCE and start_batch==0:
    print('✓ Early-stopping condition was already reached in the saved state; no additional training is needed.')
    loop_start_epoch=MAX_EPOCHS

for epoch in range(loop_start_epoch,MAX_EPOCHS):
    t0=time.time()
    if epoch==start_epoch:
        batch_start=start_batch; total=epoch_loss_sum_resume; n=epoch_sample_count_resume
    else:
        batch_start=0; total=0.; n=0
    train_loader,full_batches=make_train_loader_for_epoch(epoch,batch_start)
    model.train(); bar=tqdm(train_loader,total=full_batches-batch_start,desc=f'Epoch {epoch+1:02d}/{MAX_EPOCHS}',leave=False)
    for local_i,batch in enumerate(bar):
        batch_idx=batch_start+local_i; x=batch['image'].to(device); soft=batch['mask_soft'].to(device); optimizer.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast(enabled=amp_enabled): logits=forward_logits(x); loss=seg_loss(logits,soft)
        scaler.scale(loss).backward(); scaler.unscale_(optimizer); torch.nn.utils.clip_grad_norm_(model.parameters(),GRAD_CLIP_NORM); scaler.step(optimizer); scaler.update(); scheduler.step()
        total+=loss.item()*len(x); n+=len(x); global_step+=1; next_batch=batch_idx+1
        bar.set_postfix(loss=f'{loss.item():.4f}',batch=f'{next_batch}/{full_batches}')
        if next_batch<full_batches and next_batch%CHECKPOINT_EVERY_N_BATCHES==0:
            save_resume(epoch,next_batch,global_step,total,n,history,best_val,best_epoch,early_ref,stale)
            print(f'  ↳ GCS resume checkpoint: epoch {epoch+1}, next batch {next_batch}')

    train_loss=total/max(n,1); val=evaluate(val_loader,0.5); elapsed=time.time()-t0
    is_best=val['loss']<best_val
    if is_best:
        best_val=float(val['loss']); best_epoch=epoch+1; save_best(epoch+1,val)
    meaningful=val['loss'] < early_ref-EARLY_STOP_MIN_DELTA
    if meaningful: early_ref=float(val['loss']); stale=0
    else: stale+=1
    row={'epoch':epoch+1,'train_loss':train_loss,'val_loss':val['loss'],'val_dice_0.5':val['dice'],'val_iou_0.5':val['iou'],'lr_encoder':optimizer.param_groups[0]['lr'],'lr_new':optimizer.param_groups[1]['lr'],'seconds':elapsed,'best_val_loss':best_val,'stale_epochs':stale}; history.append(row)
    history_csv=f'{LOCAL_REPORT_DIR}/training_history.csv'; pd.DataFrame(history).to_csv(history_csv,index=False); gcs_upload(history_csv,f'{GCS_REPORT_DIR}/training_history.csv')
    save_resume(epoch+1,0,global_step,0.,0,history,best_val,best_epoch,early_ref,stale)
    flag=' ✓ BEST' if is_best else ''; mflag=' meaningful' if meaningful else ''
    print(f"Epoch {epoch+1:02d}/{MAX_EPOCHS} | train={train_loss:.4f} | val={val['loss']:.4f} | Dice@0.5={val['dice']:.4f} | IoU@0.5={val['iou']:.4f} | stale={stale}/{EARLY_STOP_PATIENCE} | {elapsed:.1f}s{flag}{mflag}")
    if stale>=EARLY_STOP_PATIENCE:
        print(f'Early stopping: {EARLY_STOP_PATIENCE} epochs without ≥{EARLY_STOP_MIN_DELTA:.4f} meaningful val-loss improvement.'); break
    start_batch=0; epoch_loss_sum_resume=0.; epoch_sample_count_resume=0

print('✓ Training phase complete | best epoch:',best_epoch,'| best val loss:',best_val)
print('Best checkpoint:',BEST_CKPT_GCS)
print('Resume checkpoint:',RESUME_CKPT_GCS)

## Step 10: TRAINING CURVES


In [ ]:
# ============================================================
# TRAINING CURVES
# ============================================================
# Plot and save:
# - Training vs validation loss
# - Validation Dice@0.5 and IoU@0.5
# ============================================================

h=pd.DataFrame(history)
if len(h)==0: raise RuntimeError('No completed epochs in history.')
plt.figure(figsize=(8,4)); plt.plot(h.epoch,h.train_loss,label='train'); plt.plot(h.epoch,h.val_loss,label='val'); plt.xlabel('Epoch'); plt.ylabel('Loss'); plt.title('SegFormer-B0 loss'); plt.legend(); plt.tight_layout(); p=f'{LOCAL_FIG_DIR}/training_loss.png'; plt.savefig(p,dpi=160,bbox_inches='tight'); plt.show(); gcs_upload(p,f'{GCS_FIG_DIR}/training_loss.png')
plt.figure(figsize=(8,4)); plt.plot(h.epoch,h['val_dice_0.5'],label='Dice@0.5'); plt.plot(h.epoch,h['val_iou_0.5'],label='IoU@0.5'); plt.xlabel('Epoch'); plt.ylabel('Score'); plt.title('Validation metrics'); plt.legend(); plt.tight_layout(); p=f'{LOCAL_FIG_DIR}/validation_metrics.png'; plt.savefig(p,dpi=160,bbox_inches='tight'); plt.show(); gcs_upload(p,f'{GCS_FIG_DIR}/validation_metrics.png')
print('✓ Curves saved to GCS')

## Step 11: VALIDATION THRESHOLD SELECTION


In [ ]:
# ============================================================
# LOAD BEST MODEL & SELECT VALIDATION THRESHOLD
# ============================================================
# Load the best validation-loss checkpoint, then sweep candidate probability
# thresholds on the VALIDATION set only.
#
# The threshold with the highest validation Dice score is selected and saved.
# The test set is not used for threshold tuning, which preserves an unbiased final test evaluation.
# ============================================================

if not os.path.exists(BEST_CKPT_LOCAL):
    if not gcs_exists(BEST_CKPT_GCS): raise RuntimeError('Best checkpoint not found on GCS.')
    gcs_download_file(BEST_CKPT_GCS,BEST_CKPT_LOCAL)
ckpt=safe_load(BEST_CKPT_LOCAL); model.load_state_dict(ckpt['model_state_dict']); model.eval(); print('✓ Loaded best epoch',ckpt['epoch'])

# -----------------------------
# Validation Threshold Sweep
# -----------------------------
@torch.no_grad()
def sweep_thresholds(loader,thresholds):
    acc={float(t):[0.,0.,0] for t in thresholds}; model.eval()
    for batch in tqdm(loader,desc='Threshold search',leave=False):
        x=batch['image'].to(device); hard=batch['mask_hard'].to(device); probs=torch.sigmoid(forward_logits(x))
        for t in thresholds:
            d,i=dice_iou(probs,hard,float(t)); acc[float(t)][0]+=d.sum().item(); acc[float(t)][1]+=i.sum().item(); acc[float(t)][2]+=len(x)
    return pd.DataFrame([{'threshold':t,'dice':v[0]/v[2],'iou':v[1]/v[2]} for t,v in acc.items()])

threshold_df=sweep_thresholds(val_loader,np.round(np.arange(0.15,0.76,0.05),2)); best_row=threshold_df.loc[threshold_df.dice.idxmax()]; BEST_THRESHOLD=float(best_row.threshold)
threshold_path=f'{LOCAL_REPORT_DIR}/threshold_search.csv'; threshold_df.to_csv(threshold_path,index=False); gcs_upload(threshold_path,f'{GCS_REPORT_DIR}/threshold_search.csv')
display(threshold_df)
print(f"✓ Best threshold={BEST_THRESHOLD:.2f} | Val Dice={best_row.dice:.4f} | Val IoU={best_row.iou:.4f}")

## Step 12: FINAL TEST EVALUATION


In [ ]:
# ============================================================
# FINAL TEST EVALUATION
# ============================================================
# Evaluate the best checkpoint exactly once on the held-out test split using
# the threshold selected from validation data.
#
# Reported outputs include:
# - Test loss
# - Dice and IoU
# - Mean / median centroid localization error
# - Fraction of predictions containing a non-empty mask
#
# Metrics and run metadata are saved to JSON and uploaded to GCS.
# ============================================================

test_metrics=evaluate(test_loader,BEST_THRESHOLD,with_centroid=True); test_metrics['best_threshold']=BEST_THRESHOLD; test_metrics['best_epoch']=int(ckpt['epoch'])
metrics_path=f'{LOCAL_REPORT_DIR}/test_metrics.json'
with open(metrics_path,'w') as f: json.dump(test_metrics,f,indent=2)
gcs_upload(metrics_path,f'{GCS_REPORT_DIR}/test_metrics.json')
print('============================================================')
print('FINAL TEST RESULTS')
print('============================================================')
for k,v in test_metrics.items(): print(f'{k:26s}: {v}')
print('============================================================')

## Step 13: PREDICTION VISUALIZATION


In [ ]:
# ============================================================
# PREDICTION VISUALIZATION
# ============================================================
# Draw a fixed, reproducible sample of test examples and compare:
# 1. RGB visualization derived from B4/B3/B2
# 2. Hard target mask
# 3. Soft sigma-1 training target
# 4. Predicted probability map
# 5. Final binary mask at BEST_THRESHOLD
# ============================================================

def rgb_display(x):
    rgb=np.stack([x[2],x[1],x[0]],axis=-1).astype(np.float32); p1=np.percentile(rgb,1); p99=np.percentile(rgb,99)
    return np.clip((rgb-p1)/(p99-p1+1e-6),0,1)

@torch.no_grad()
def predict_item(ds,idx):
    s=ds[idx]; prob=torch.sigmoid(forward_logits(s['image'].unsqueeze(0).to(device)))[0,0].cpu().numpy(); return s,prob

# -----------------------------
# Reproducible Test-Sample Visualization
# -----------------------------
rng=np.random.default_rng(FIXED_VIS_SEED); n=min(N_VIS_SAMPLES,len(test_ds)); ids=rng.choice(len(test_ds),size=n,replace=False)
fig,axes=plt.subplots(n,5,figsize=(15,3*n)); axes=np.atleast_2d(axes)
for r,idx in enumerate(ids):
    s,prob=predict_item(test_ds,int(idx)); x=s['image'].numpy(); hard=s['mask_hard'].numpy()[0]; soft=s['mask_soft'].numpy()[0]; pred=(prob>=BEST_THRESHOLD).astype(np.float32)
    axes[r,0].imshow(rgb_display(x)); axes[r,0].set_title('RGB display')
    axes[r,1].imshow(hard,vmin=0,vmax=1); axes[r,1].set_title('Hard label')
    axes[r,2].imshow(soft,vmin=0,vmax=1); axes[r,2].set_title('Soft sigma1')
    axes[r,3].imshow(prob,vmin=0,vmax=1); axes[r,3].set_title('Pred probability')
    axes[r,4].imshow(pred,vmin=0,vmax=1); axes[r,4].set_title(f'Pred ≥ {BEST_THRESHOLD:.2f}')
    axes[r,0].set_ylabel(s['meta']['sample_id'],fontsize=8)
    for a in axes[r]: a.axis('off')
fig.suptitle('SegFormer-B0 — prediction vs label'); plt.tight_layout(); fig_path=f'{LOCAL_FIG_DIR}/prediction_vs_label.png'; plt.savefig(fig_path,dpi=160,bbox_inches='tight'); plt.show(); gcs_upload(fig_path,f'{GCS_FIG_DIR}/prediction_vs_label.png')
print('✓ Prediction comparison saved to GCS')

## Step 14: MODEL EXPORT


In [ ]:
# ============================================================
# EXPORT FINAL MODEL & INFERENCE METADATA
# ============================================================
# Export the trained SegFormer in Hugging Face format together with all
# preprocessing information required for reproducible inference.
#
# The inference config records:
# - Image size and exact 11-channel order
# - Selected probability threshold
# - Train-derived normalization parameters
#
# Finally, the Hugging Face model, summary JSON, reports, and figures are
# synchronized to the experiment's Google Cloud Storage directory.
# ============================================================

model.config.num_channels=N_CHANNELS; model.config.num_labels=1; model.config.id2label={0:'geoglyph'}; model.config.label2id={'geoglyph':0}
model.save_pretrained(LOCAL_HF_DIR,safe_serialization=True)
# -----------------------------
# Save Inference Contract
# -----------------------------
inference_config={'run_name':RUN_NAME,'image_size':IMAGE_SIZE,'channel_order':ZARR_CHANNEL_ORDER,'label_key':LABEL_KEY,'threshold':BEST_THRESHOLD,'normalization_params':stage2_norm_params,'note':'Input must remain [B2,B3,B4,B8,B11,B12,DEM,Slope,NDVI,NDWI,BSI].'}
with open(f'{LOCAL_HF_DIR}/geoglyph_inference_config.json','w') as f: json.dump(inference_config,f,indent=2)
# -----------------------------
# Upload Model Artifacts and Final Run Summary
# -----------------------------
gcs_rsync(LOCAL_HF_DIR,GCS_HF_DIR)
summary={'best_epoch':int(ckpt['epoch']),'best_threshold':BEST_THRESHOLD,'test_metrics':test_metrics,'best_checkpoint_gcs':BEST_CKPT_GCS,'resume_checkpoint_gcs':RESUME_CKPT_GCS,'hf_model_gcs':GCS_HF_DIR,'reports_gcs':GCS_REPORT_DIR,'figures_gcs':GCS_FIG_DIR}
summary_path=f'{LOCAL_REPORT_DIR}/final_summary.json'
with open(summary_path,'w') as f: json.dump(summary,f,indent=2)
gcs_upload(summary_path,f'{GCS_REPORT_DIR}/final_summary.json')
print('============================================================')
print('SEGFORMER RUN COMPLETE')
print('Best checkpoint:',BEST_CKPT_GCS)
print('Resume checkpoint:',RESUME_CKPT_GCS)
print('HF model:',GCS_HF_DIR)
print('Reports:',GCS_REPORT_DIR)
print('Figures:',GCS_FIG_DIR)
print('============================================================')